# Data Analysis Agent Demo

This notebook demonstrates how to use the Data Analysis Agent for automating database queries from JIRA tickets. We'll walk through the process of setting up the agent, configuring it, and running it on a sample task.

## 1. Setup and Dependencies

First, let's install the necessary dependencies if you haven't already.

In [ ]:
# Install dependencies if needed
# !pip install -r requirements.txt

# Import required libraries
import os
import sys
import logging
import pandas as pd
from pathlib import Path
import yaml

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Add project root to path to make imports work
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Check Python version
import platform
print(f"Python version: {platform.python_version()}")

## 2. Configuration

Let's load the configuration for our agent. You might need to update the connection details for your own environment.

In [ ]:
# Load configuration
config_path = Path("config/config.yaml")

def load_config(config_path):
    with open(config_path, 'r') as file:
        return yaml.safe_load(file)

# Load and display config (hiding sensitive information)
config = load_config(config_path)
print("Agent configuration loaded:")
print(f"- Max retries: {config['agent']['max_retries']}")
print(f"- LLM model: {config['agent']['llm']['model_name']}")
print(f"- Database connection: {'configured' if config['database']['connection_string'] else 'missing'}")
print(f"- JIRA project key: {config['jira']['project_key']}")

## 3. Setting up the LLM

We need to configure access to the OpenAI API for our agent to generate SQL queries, validate them, and generate insights.

In [ ]:
# Set up OpenAI API access
import os
from dotenv import load_dotenv

# Load environment variables from .env file if present
load_dotenv()

# Check if OpenAI API key is set
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    print("WARNING: OPENAI_API_KEY environment variable not set!")
    # For demo purposes, you can set it here, but never commit this to version control
    # os.environ["OPENAI_API_KEY"] = "your-api-key-here"
else:
    print("OpenAI API key is configured.")

## 4. Exploring the Database

Let's connect to the database and examine its schema. This helps us understand what data we're working with.

In [ ]:
# Import the database client
from src.clients.db_client import DatabaseClient

# Initialize database client with a connection string
# For demo purposes, we might use a SQLite database or a test/staging database
# Replace this with your actual connection string or use the one from config

# Use a demo connection string if needed
demo_connection_string = "sqlite:///demo_database.db"
# Uncomment below to use the config connection
# demo_connection_string = config['database']['connection_string']

try:
    # Create database client
    db_client = DatabaseClient(demo_connection_string)
    
    # Get database schema
    schema = db_client.get_database_schema()
    
    # Display schema
    print("Database Schema:")
    for table_name, columns in schema.items():
        print(f"\nTable: {table_name}")
        for column in columns:
            print(f"  - {column['column_name']} ({column['data_type']})")
except Exception as e:
    print(f"Error connecting to database: {str(e)}")
    print("Using mock schema for demonstration")
    
    # Create a mock schema for demonstration
    schema = {
        "sales": [
            {"column_name": "id", "data_type": "integer"},
            {"column_name": "date", "data_type": "date"},
            {"column_name": "product_id", "data_type": "integer"},
            {"column_name": "quantity", "data_type": "integer"},
            {"column_name": "price", "data_type": "numeric"}
        ],
        "products": [
            {"column_name": "id", "data_type": "integer"},
            {"column_name": "name", "data_type": "character varying"},
            {"column_name": "category", "data_type": "character varying"},
            {"column_name": "cost", "data_type": "numeric"}
        ],
        "customers": [
            {"column_name": "id", "data_type": "integer"},
            {"column_name": "name", "data_type": "character varying"},
            {"column_name": "email", "data_type": "character varying"},
            {"column_name": "country", "data_type": "character varying"}
        ]
    }
    
    # Display the mock schema
    print("Mock Database Schema:")
    for table_name, columns in schema.items():
        print(f"\nTable: {table_name}")
        for column in columns:
            print(f"  - {column['column_name']} ({column['data_type']})")

## 5. Creating a Test JIRA Ticket

For demonstration purposes, let's create a simulated JIRA ticket with a data analysis task. In a real scenario, this would come from your JIRA instance.

In [ ]:
# Import required models
from src.models.schemas import JiraTicket, TicketStatus

# Create a simulated JIRA ticket with a data analysis task
test_ticket = JiraTicket(
    ticket_id="DATA-42",
    summary="Sales performance analysis by product category",
    description="Please analyze our sales data from last month. Compare the performance of different product categories and identify the top-selling products in each category.",
    status=TicketStatus.OPEN
)

print(f"Test Ticket: {test_ticket.ticket_id} - {test_ticket.summary}")
print(f"Description: {test_ticket.description}")

## 6. Initialize the Data Analysis Agent

Now let's initialize the agent with our configuration.

In [ ]:
# Import agent
from src.agent.agent import DataAnalysisAgent
from langchain.llms import OpenAI
from src.tools.sql_tool import SQLTool
from src.tools.validator_tool import ValidatorTool
from src.tools.insight_tool import InsightTool
from src.models.schemas import SQLQuery, ValidationResult, QueryResult, BusinessInsight

# For demo purposes, we'll create a simplified agent with minimal dependencies
# This avoids the need for actual JIRA connection and full database access

class DemoAgent:
    def __init__(self, schema):
        # Initialize the LLM
        self.llm = OpenAI(
            model_name="gpt-4",  # Use your preferred model
            temperature=0.1,
            max_tokens=1500
        )
        
        # Initialize tools
        self.sql_tool = SQLTool(llm=self.llm)
        self.validator_tool = ValidatorTool(llm=self.llm, schema_dict=schema)
        self.insight_tool = InsightTool(llm=self.llm)
        
        # Store schema
        self.schema = schema
        
    def generate_sql(self, task_description):
        """Generate SQL query from task description."""
        return self.sql_tool.generate_query(task_description, self.schema)
    
    def validate_sql(self, sql_query, task_description):
        """Validate the generated SQL query."""
        return self.validator_tool.validate_sql(sql_query, task_description)
    
    def simulate_query_results(self, sql_query):
        """Simulate query results for demonstration."""
        # This is a simplified simulation - in real use, we'd execute the query
        # Create sample data based on the query
        # Here we'll just provide mock data relevant to our test task
        
        if "product" in sql_query.query.lower() and "category" in sql_query.query.lower():
            # Simulate product category analysis
            data = [
                {"category": "Electronics", "total_sales": 125000, "avg_price": 450},
                {"category": "Clothing", "total_sales": 89000, "avg_price": 65},
                {"category": "Home Goods", "total_sales": 67000, "avg_price": 120},
                {"category": "Books", "total_sales": 34000, "avg_price": 22},
                {"category": "Toys", "total_sales": 28000, "avg_price": 35}
            ]
            
            return QueryResult(
                data=data,
                row_count=len(data),
                column_names=["category", "total_sales", "avg_price"],
                execution_time_ms=250.0
            )
        else:
            # Default mock data
            return QueryResult(
                data=[{"result": "No specific mock data available for this query"}],
                row_count=1,
                column_names=["result"],
                execution_time_ms=100.0
            )
    
    def generate_insights(self, task_description, query_result):
        """Generate business insights from query results."""
        return self.insight_tool.generate_insights(task_description, query_result)

# Create our demo agent
demo_agent = DemoAgent(schema)
print("Demo agent initialized successfully!")

## 7. Process the Test Ticket

Now let's process our test ticket through the agent's workflow.

In [ ]:
# Extract task from ticket
task_description = test_ticket.description
print(f"Processing task: {task_description}")

# Step 1: Generate SQL query
print("\n--- STEP 1: Generate SQL Query ---")
try:
    sql_query = demo_agent.generate_sql(task_description)
    print(f"SQL Query Generated: \n{sql_query.query}")
    print(f"Description: {sql_query.description}")
    print(f"Tables used: {', '.join(sql_query.tables_used)}")
except Exception as e:
    print(f"Error generating SQL: {str(e)}")
    sql_query = None

In [ ]:
# Step 2: Validate SQL query
print("\n--- STEP 2: Validate SQL Query ---")
if sql_query:
    try:
        validation_result = demo_agent.validate_sql(sql_query, task_description)
        print(f"SQL Validation: {'✓ Valid' if validation_result.is_valid else '✗ Invalid'}")
        if validation_result.errors:
            print("Errors:")
            for error in validation_result.errors:
                print(f"- {error}")
        if validation_result.warnings:
            print("Warnings:")
            for warning in validation_result.warnings:
                print(f"- {warning}")
    except Exception as e:
        print(f"Error validating SQL: {str(e)}")
        validation_result = None
else:
    print("Skipping validation - no SQL query available")

In [ ]:
# Step 3: Simulate query execution
print("\n--- STEP 3: Execute Query ---")
if sql_query:
    try:
        query_result = demo_agent.simulate_query_results(sql_query)
        print(f"Query executed successfully.")
        print(f"Returned {query_result.row_count} rows with columns: {', '.join(query_result.column_names)}")
        
        # Display a sample of the results
        if query_result.data:
            print("\nSample results:")
            df = pd.DataFrame(query_result.data)
            display(df.head())
    except Exception as e:
        print(f"Error executing query: {str(e)}")
        query_result = None
else:
    print("Skipping execution - no SQL query available")

In [ ]:
# Step 4: Generate insights
print("\n--- STEP 4: Generate Insights ---")
if query_result:
    try:
        insights = demo_agent.generate_insights(task_description, query_result)
        print("Insights generated successfully:")
        print(f"\nSummary: {insights.summary}")
        
        print("\nKey Points:")
        for point in insights.key_points:
            print(f"- {point}")
            
        if insights.recommendations:
            print("\nRecommendations:")
            for rec in insights.recommendations:
                print(f"- {rec}")
    except Exception as e:
        print(f"Error generating insights: {str(e)}")
        insights = None
else:
    print("Skipping insights - no query results available")

## 8. Complete Agent Workflow

In the previous sections, we stepped through each component of the agent process. Now let's simulate the full workflow from start to finish.

In [ ]:
# Define a function that simulates the full agent workflow
def process_ticket_workflow(ticket):
    print(f"Processing ticket: {ticket.ticket_id} - {ticket.summary}")
    
    # Extract task description
    task_description = ticket.description
    print(f"\nTask: {task_description}")
    
    # Step 1: Generate SQL
    print("\n1. Generating SQL query...")
    sql_query = demo_agent.generate_sql(task_description)
    print(f"SQL generated: \n{sql_query.query}")
    
    # Step 2: Validate SQL
    print("\n2. Validating SQL query...")
    validation_result = demo_agent.validate_sql(sql_query, task_description)
    
    if not validation_result.is_valid:
        print("SQL validation failed:")
        for error in validation_result.errors:
            print(f"- {error}")
        print("Workflow would retry or fail here in production.")
        return None
    
    print("SQL validation passed!")
    
    # Step 3: Execute query
    print("\n3. Executing query...")
    query_result = demo_agent.simulate_query_results(sql_query)
    print(f"Query executed - returned {query_result.row_count} rows.")
    
    # Step 4: Generate insights
    print("\n4. Generating business insights...")
    insights = demo_agent.generate_insights(task_description, query_result)
    
    print("\n----- RESULTS -----")
    print(f"Summary: {insights.summary}")
    
    print("\nKey Points:")
    for point in insights.key_points:
        print(f"- {point}")
        
    if insights.recommendations:
        print("\nRecommendations:")
        for rec in insights.recommendations:
            print(f"- {rec}")
    
    print("\n5. In a production environment, these results would be posted to JIRA ticket.")
    
    return insights

# Create a new test ticket with a different task
advanced_ticket = JiraTicket(
    ticket_id="DATA-43",
    summary="Customer retention analysis",
    description="Analyze our customer data to identify patterns of customer retention. Which customer segments have the highest retention rates, and which products are most effective at retaining customers?",
    status=TicketStatus.OPEN
)

# Process the ticket
process_ticket_workflow(advanced_ticket)

## 9. Next Steps

Now that you've seen how the Data Analysis Agent works, you can:

1. **Configure the agent for your environment**:
   - Update the database connection details in `config/config.yaml`
   - Configure your JIRA credentials

2. **Run the agent on real tickets**:
   ```python
   from src.agent.agent import DataAnalysisAgent
   agent = DataAnalysisAgent()
   agent.process_open_tickets(max_tickets=5)
   ```

3. **Extend the agent capabilities**:
   - Add new validation rules to the `ValidatorTool`
   - Enhance the SQL generation with custom templates
   - Add visualization capabilities to the insights

4. **Schedule the agent**:
   - Set up a cron job or scheduler to run the agent periodically
   - Integrate with your workflows or CI/CD pipelines

## 10. Conclusion

The Data Analysis Agent demonstrates how LangGraph can be used to create a powerful AI agent that automates routine data analysis tasks. By combining SQL generation, validation, execution, and insight generation in a feedback loop, the agent can handle complex analytical tasks with minimal human intervention.

Key benefits:
- Reduces manual effort for routine data queries
- Ensures consistent, high-quality analysis
- Provides a flexible framework that can be extended for other use cases
- Maintains a human-in-the-loop approach with JIRA integration

This notebook provided a demonstration of the core functionality. For production use, you would need to properly configure database access, JIRA integration, and potentially add additional error handling and security measures.